# Ignite-3B Session S03 - Cond C gen 0-2 (same-model RSI MAIN)

**Goal**: run Cond C outer loop for gens 0-2 (8 candidates each, ~150 steps).

Setup:
1. Accelerator = GPU T4 x2
2. Internet ON
3. Persistence = Variables and Files
4. Kaggle Secret HF_TOKEN write access

Push v_0..v_2 to HF privado. Resume-safe via `--resume`.

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
HF_REPO = 'iterate-labs-ai/ignite-3b-cond-c'
BENCH = 'math'
BENCH_NAME = 'omni_math'
DATASET_TRAIN = 'data/ignite/omni_math_train.jsonl'
DATASET_DEV = 'data/ignite/omni_math_dev.jsonl'
DATASET_VAL = 'data/ignite/omni_math_val.jsonl'
OUT_ROOT = '/kaggle/working/cond_C_run'
GENS = 1   # session 12h Kaggle: 1 gen, small subset
CANDS = 3   # 3 candidates per gen
STEPS = 50  # 50 GRPO steps each
print(f'S03 - Cond C - gens={GENS} cands={CANDS} steps={STEPS}')

In [ ]:
# torch 2.6.0 pra trl.GRPOTrainer (FSDPModule import). Suporta SM 6.0/7.5/8.0+.
!pip install -q "torch==2.6.0" "torchvision==0.21.0" --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U 'transformers>=4.46.0,<5' 'peft>=0.13.0' 'datasets>=3.0.0' 'accelerate>=1.0.0' 'trl>=0.14.0,<0.20' 'math-verify>=0.5.2' 'latex2sympy2' 'sympy' 'scipy' 'antlr4-python3-runtime==4.11' huggingface_hub

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/caracal-1'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 's07-hybrid-agentic',
                    'https://github.com/iterate-labs-ai/caracal-1.git',
                    '/kaggle/working/caracal-1'], check=True)
os.chdir('/kaggle/working/caracal-1')
print(subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print('HF login OK')
except Exception as e:
    print(f'HF login skipped ({e}) - will not push to HF')

In [ ]:
import subprocess, os, json, random
if not os.path.exists('data/ignite/omni_math.jsonl'):
    subprocess.run(['python', '-m', 'data.ignite.build_omni_math'], check=True)
rows = [json.loads(line) for line in open('data/ignite/omni_math.jsonl')]
random.Random(42).shuffle(rows)
# Small subset pra caber em 12h T4 x2 kernel (train 200 rows, dev/val 100 cada)
rows = rows[:400]
n_train, n_dev = 200, 100
with open(DATASET_TRAIN, 'w') as f: [f.write(json.dumps(r) + '\n') for r in rows[:n_train]]
with open(DATASET_DEV, 'w') as f: [f.write(json.dumps(r) + '\n') for r in rows[n_train:n_train+n_dev]]
with open(DATASET_VAL, 'w') as f: [f.write(json.dumps(r) + '\n') for r in rows[n_train+n_dev:]]
print(f'train={n_train} dev={n_dev} val={len(rows)-n_train-n_dev}')

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'train.ignite.C_rsi_outer',
       '--base', BASE_MODEL,
       '--dataset-train', DATASET_TRAIN,
       '--dataset-dev', DATASET_DEV,
       '--dataset-val', DATASET_VAL,
       '--bench', BENCH,
       '--bench-name', BENCH_NAME,
       '--gens', str(GENS),
       '--cands', str(CANDS),
       '--steps', str(STEPS),
       '--out', OUT_ROOT]
if os.environ.get('HF_TOKEN'):
    cmd += ['--hf-repo', HF_REPO]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
log = [json.loads(line) for line in open(f'{OUT_ROOT}/log.jsonl')]
gens = [r for r in log if r.get('type') == 'generation']
print('accepted generations:')
for g in gens:
    print(f"  gen{g['gen']}: val_r={g['val_r']:.4f} delta={g['delta_pp']:.4f}")
print(f"\nHF repo: https://huggingface.co/{HF_REPO}")